# DNS attacks (spoofing, cache poisoning)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 3.1.b: DNS resolver race classifier

**Skill:** Analyze Risk, Detect Attacks · **Maps to:** 3.1.A, 3.1.B · **Language:** Python 3 (pytest)

## Problem
You are building triage logic for a resolver reply log at Crestfield Analytics. For each
outstanding query, the log records the replies that arrived, and, for each reply, who sent it
(`"authoritative"` or `"attacker"`), and the `tx_id`, `port`, and `qname` it carried. Classify each
query's outcome:

- **"REJECT"**, no reply is a valid candidate (none match the query's `tx_id`, `port`, AND
  `qname` all three), so nothing is cached. This also covers the case of zero replies.
- **"WIN"**, among the valid candidates (replies matching all three fields), the one with the
  smallest `arrival` number (i.e. the first to arrive) has `"from": "authoritative"`.
- **"POISON"**, among the valid candidates, the first-arriving one has `"from": "attacker"`.

## Function contract
Implement `solve(queries)` in the code editor (see the Reference section below).

## Input
- `queries`: `list[dict]`, each with:
  - `"qname"`: `str`, the domain name queried
  - `"tx_id"`: `int`, the transaction ID the resolver sent
  - `"port"`: `int`, the source port the resolver sent the query from
  - `"replies"`: `list[dict]`, each with `"from"` (`"authoritative"` or `"attacker"`),
    `"tx_id"` (`int`), `"port"` (`int`), `"qname"` (`str`), `"arrival"` (`int`, lower = earlier)

## Output
A `list[str]` of classifications, **one per input query, in the same order as the input**. Each
entry is one of `"WIN"`, `"POISON"`, `"REJECT"`.

## Examples
```python
queries = [
    {
        "qname": "bank.example", "tx_id": 48291, "port": 33412,
        "replies": [
            {"from": "attacker", "tx_id": 48291, "port": 33412, "qname": "bank.example", "arrival": 1},
            {"from": "authoritative", "tx_id": 48291, "port": 33412, "qname": "bank.example", "arrival": 2},
        ],
    },
]
solve(queries)  ->  ["POISON"]

queries2 = [
    {
        "qname": "shop.example", "tx_id": 11020, "port": 40500,
        "replies": [
            {"from": "attacker", "tx_id": 99999, "port": 40500, "qname": "shop.example", "arrival": 1},
            {"from": "authoritative", "tx_id": 11020, "port": 40500, "qname": "shop.example", "arrival": 2},
        ],
    },
]
solve(queries2)  ->  ["WIN"]
```

## Constraints
- Deterministic, offline, standard library only.
- A reply is a "valid candidate" only if `reply["tx_id"] == query["tx_id"]` AND
  `reply["port"] == query["port"]` AND `reply["qname"] == query["qname"]`.
- Among valid candidates, only the one with the minimum `arrival` value matters (test data never
  has ties on the minimum `arrival`).
- Replies may be given in any order in the list; use the `arrival` field, not list position, to
  determine which arrived first.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(queries: list[dict]) -> list[str]
```

### Input (`queries`)
`list[dict]` where each dict has:
- `"qname"`: `str`, domain name queried
- `"tx_id"`: `int`, transaction ID the resolver sent
- `"port"`: `int`, source port the resolver sent the query from
- `"replies"`: `list[dict]`, each with:
  - `"from"`: `str`, `"authoritative"` or `"attacker"`
  - `"tx_id"`: `int`
  - `"port"`: `int`
  - `"qname"`: `str`
  - `"arrival"`: `int`, arrival order, lower arrives earlier

### Output (`result`)
A `list[str]` the same length as `queries`, in the **same order as the input**, where each element
is one of `"WIN"`, `"POISON"`, `"REJECT"`.

### Classification rules
1. A reply is a **valid candidate** iff its `tx_id`, `port`, and `qname` all equal the query's
   `tx_id`, `port`, and `qname`.
2. If there are no valid candidates (including an empty `replies` list), the result is `"REJECT"`.
3. Otherwise, take the valid candidate with the smallest `arrival` value (the first to arrive). If
   its `"from"` is `"authoritative"`, the result is `"WIN"`; if `"attacker"`, the result is
   `"POISON"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output order matches input order exactly.
- Test data never has two valid candidates with an identical `arrival` value.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 3.1.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import List, Dict


def solve(queries: List[Dict]) -> List[str]:
    """Classify each DNS query's resolver race outcome as WIN, POISON, or REJECT.

    Rules (see the Reference section below):
      1. A reply is a "valid candidate" iff tx_id, port, and qname all match the query.
      2. No valid candidates -> REJECT.
      3. Otherwise the valid candidate with the smallest 'arrival' wins the race:
         from == "authoritative" -> WIN, from == "attacker" -> POISON.
    """
    result = []
    for q in queries:
        candidates = [
            r for r in q["replies"]
            if r["tx_id"] == q["tx_id"] and r["port"] == q["port"] and r["qname"] == q["qname"]
        ]
        if not candidates:
            result.append("REJECT")
            continue
        winner = min(candidates, key=lambda r: r["arrival"])
        result.append("WIN" if winner["from"] == "authoritative" else "POISON")
    return result


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _r(frm, tx_id, port, qname, arrival):
    return {"from": frm, "tx_id": tx_id, "port": port, "qname": qname, "arrival": arrival}


def _q(qname, tx_id, port, replies):
    return {"qname": qname, "tx_id": tx_id, "port": port, "replies": replies}


# ---- visible cases -------------------------------------------------------

def test_poison_attacker_arrives_first():
    queries = [_q("bank.example", 48291, 33412, [
        _r("attacker", 48291, 33412, "bank.example", 1),
        _r("authoritative", 48291, 33412, "bank.example", 2),
    ])]
    assert impl.solve(queries) == ["POISON"]


def test_win_attacker_wrong_tx_id():
    queries = [_q("shop.example", 11020, 40500, [
        _r("attacker", 99999, 40500, "shop.example", 1),
        _r("authoritative", 11020, 40500, "shop.example", 2),
    ])]
    assert impl.solve(queries) == ["WIN"]


def test_reject_wrong_port():
    queries = [_q("mail.example", 5551, 21999, [
        _r("attacker", 5551, 9999, "mail.example", 1),
    ])]
    assert impl.solve(queries) == ["REJECT"]


def test_win_legit_arrives_first():
    queries = [_q("vpn.example", 7000, 51234, [
        _r("authoritative", 7000, 51234, "vpn.example", 1),
        _r("attacker", 7000, 51234, "vpn.example", 2),
    ])]
    assert impl.solve(queries) == ["WIN"]


def test_reject_wrong_qname():
    queries = [_q("cdn.example", 6002, 60111, [
        _r("attacker", 6002, 60111, "cdn-fake.example", 1),
    ])]
    assert impl.solve(queries) == ["REJECT"]


def test_empty_replies_is_reject():
    queries = [_q("sync.example", 3300, 22222, [])]
    assert impl.solve(queries) == ["REJECT"]


def test_empty_queries_list():
    assert impl.solve([]) == []


def test_mixed_batch_order_preserved():
    queries = [
        _q("z9.example", 1, 1, []),
        _q("front.example", 5, 5, [_r("authoritative", 5, 5, "front.example", 1)]),
        _q("vault.example", 9, 9, [_r("attacker", 9, 9, "vault.example", 1)]),
    ]
    assert impl.solve(queries) == ["REJECT", "WIN", "POISON"]


# ---- hidden / edge cases -------------------------------------------------

def test_flood_of_guessed_ids_all_miss_then_real_wins():
    """Kaminsky-style flood: multiple attacker guesses miss tx_id, real reply wins eventually."""
    queries = [_q("portal.example", 8123, 15000, [
        _r("attacker", 100, 15000, "portal.example", 1),
        _r("attacker", 200, 15000, "portal.example", 2),
        _r("authoritative", 8123, 15000, "portal.example", 3),
    ])]
    assert impl.solve(queries) == ["WIN"]


def test_flood_where_one_guess_hits_first():
    """One attacker guess matches all three fields and arrives before the real reply -> POISON."""
    queries = [_q("hr.example", 9500, 44210, [
        _r("attacker", 9500, 44210, "hr.example", 1),
        _r("attacker", 9500, 44210, "hr.example", 2),
        _r("authoritative", 9500, 44210, "hr.example", 3),
    ])]
    assert impl.solve(queries) == ["POISON"]


def test_replies_out_of_list_order_still_use_arrival_field():
    """Replies list is not sorted by arrival; arrival field (not list position) determines winner."""
    queries = [_q("late.example", 42, 42, [
        _r("authoritative", 42, 42, "late.example", 5),
        _r("attacker", 42, 42, "late.example", 2),
    ])]
    assert impl.solve(queries) == ["POISON"]


def test_only_qname_mismatch_among_otherwise_valid_replies():
    queries = [_q("real.example", 3, 3, [
        _r("attacker", 3, 3, "notreal.example", 1),
        _r("authoritative", 3, 3, "real.example", 2),
    ])]
    assert impl.solve(queries) == ["WIN"]


def test_multiple_valid_candidates_from_both_sides_lowest_arrival_wins():
    queries = [_q("multi.example", 7, 7, [
        _r("authoritative", 7, 7, "multi.example", 3),
        _r("attacker", 7, 7, "multi.example", 1),
        _r("attacker", 7, 7, "multi.example", 2),
    ])]
    assert impl.solve(queries) == ["POISON"]


def test_large_batch_deterministic():
    queries = []
    expected = []
    for i in range(20):
        if i % 3 == 0:
            queries.append(_q(f"q{i}.example", i, i, []))
            expected.append("REJECT")
        elif i % 3 == 1:
            queries.append(_q(f"q{i}.example", i, i, [_r("authoritative", i, i, f"q{i}.example", 1)]))
            expected.append("WIN")
        else:
            queries.append(_q(f"q{i}.example", i, i, [_r("attacker", i, i, f"q{i}.example", 1)]))
            expected.append("POISON")
    assert impl.solve(queries) == expected


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
